In [ ]:
import os
os.chdir('../../../../..')

In [ ]:
import numpy as np
import hdbscan
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from mace.calculators import mace_off
from rdkit import Chem
import gc
import numpy as np
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import umap
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolTransforms
from dscribe.descriptors import SOAP
from ase import Atoms
import polars as pl
from scripts.qm9.chemprop import CheMeleonFingerprint
from dscribe.descriptors import SOAP
from rdkit import Chem
from tqdm import tqdm
import numpy as np
import polars as pl
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
import polars as pl
from ase import Atoms
from dscribe.descriptors import SOAP
from rdkit import Chem
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
import numpy as np
import polars as pl
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from sklearn.cluster import AgglomerativeClustering, SpectralClustering, DBSCAN
from kmedoids import KMedoids
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
from umap import UMAP
from dscribe.kernels import REMatchKernel

from src.datasets import QM9Dataset
from src.helper_functions import plot_molecules_with_py3dmol, create_chemiscope_viewer, plot_distance_matrix_projection, evaluate_distance_matrix_clustering_sweep, average_numeric_by_cluster, evaluate_hdbscan_grid, get_isomers
from src.non_euclidean import Riemann, _feature_matrices_from_df

projection_method = "MDS"

In [12]:
qm9 = QM9Dataset(limit=5_000, descriptors=["soap", "mace"])
df = qm9.load()

2026-06-17 19:44:11.221 | INFO     | src.datasets:_load_full_qm9_df:935 - Loading cached full QM9 dataset from: data/QM9/dataset_cleaned.parquet
2026-06-17 19:44:11.691 | INFO     | src.datasets:_sample_qm9_df:1119 - QM9 sampling complete: strategy=stratified, requested_limit=5000, returned_rows=5000, sampling on columns=['num_atoms', 'gap'].
2026-06-17 19:44:11.692 | INFO     | src.datasets:_add_requested_descriptors:293 - Applying requested QM9 descriptors to sampled dataframe (rows=5000).
2026-06-17 19:44:11.699 | INFO     | src.features:compute_soap_outputs:395 - Computing SOAP (rcut=6.0, nmax=8, lmax=6, normalize=True)...
2026-06-17 19:44:16.122 | SUCCESS  | src.datasets:add_soap:1307 - Added SOAP embeddings and matrices.
2026-06-17 19:44:16.123 | INFO     | src.features:compute_mace_outputs:679 - Computing MACE embeddings (model=medium, batch_size=32)...
/Users/karlfindhansen/thesis/Anomaly-Detection-in-Molecular-and-Materials-Datasets/.venv/lib/python3.12/site-packages/mace/calc

Using MACE-OFF23 MODEL for MACECalculator with /Users/karlfindhansen/.cache/mace/MACE-OFF23_medium.model
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.


2026-06-17 19:49:13.533 | SUCCESS  | src.datasets:add_mace:1350 - Added MACE embeddings and matrices.
2026-06-17 19:49:13.534 | INFO     | src.datasets:_add_requested_descriptors:304 - Added descriptor column(s): ['mace_embedding', 'mace_matrix', 'soap_embedding', 'soap_matrix']
2026-06-17 19:49:13.543 | INFO     | src.datasets:_load_with_descriptor_filter:975 - QM9 descriptor null-filtering complete: attempts=1, requested_limit=5000, returned_rows=5000, base_rows=5000.


In [ ]:
from __future__ import annotations
import numpy as np
import polars as pl
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy.spatial.distance import pdist
from scipy.stats import spearmanr
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor, NearestNeighbors
from sklearn.model_selection import KFold, cross_val_score
import umap  # umap-learn


def _build_atoms(coords, Z):
    from ase import Atoms

    coords = np.asarray(coords, dtype=float).reshape(-1, 3)
    Z = np.asarray(Z, dtype=int).ravel()
    return Atoms(numbers=Z, positions=coords)


def add_soap_column(
    df: pl.DataFrame,
    r_cut: float = 6.0,          # Table 9 used 4.5 and 6.0 -- match what Table 10 used
    n_max: int = 8,
    l_max: int = 6,
    sigma: float = 0.5,
    species=("H", "C", "N", "O", "F"),
    col: str = "soap",
) -> pl.DataFrame:
    """Per-atom SOAP via DScribe. NB: arg names / compression API are version
    sensitive (DScribe >= 1.2 uses r_cut/n_max/l_max). Verify D matches your
    thesis (252) before trusting the comparison."""
    from dscribe.descriptors import SOAP

    soap = SOAP(
        species=list(species),
        r_cut=r_cut,
        n_max=n_max,
        l_max=l_max,
        sigma=sigma,
        periodic=False,
        compression={"mode": "mu2"},  # Darby et al. 2022 mu2 compression
        sparse=False,
    )
    feats = [
        soap.create(_build_atoms(c, z)).tolist()
        for c, z in zip(df["coordinates"].to_list(), df["atomic_numbers"].to_list())
    ]
    return df.with_columns(pl.Series(col, feats))


def add_mace_column(
    df: pl.DataFrame,
    model: str = "medium",
    device: str = "cpu",
    col: str = "mace",
) -> pl.DataFrame:
    """Per-atom MACE-OFF23 invariant descriptors (l=0, all layers concatenated
    => 256 for medium). Matches section 2.4.3 / 3.3.2."""
    from mace.calculators import mace_off

    calc = mace_off(model=model, device=device, default_dtype="float64")
    feats = []
    for c, z in zip(df["coordinates"].to_list(), df["atomic_numbers"].to_list()):
        d = calc.get_descriptors(_build_atoms(c, z), invariants_only=True, num_layers=-1)
        feats.append(np.asarray(d, dtype=float).tolist())
    return df.with_columns(pl.Series(col, feats))


# ======================================================================
# 2. EMBEDDING BUILDERS  (depend on your code)
# ======================================================================
def mean_pool(df: pl.DataFrame, descriptor: str) -> np.ndarray:
    """1st-moment baseline: mean over atomic rows -> (N_mol, D)."""
    mats = _feature_matrices_from_df(df, descriptor)
    return np.vstack([np.asarray(X, dtype=float).mean(axis=0) for X in mats])


def tangent(df: pl.DataFrame, descriptor: str, pca: bool = True) -> np.ndarray:
    """2nd-moment: log-Euclidean tangent vectors via your Riemann class."""
    if Riemann is None:
        raise RuntimeError("Riemann not imported -- fix the import at the top.")
    return Riemann.vectorized_spd_matrices(df, descriptor=descriptor, pca=pca)


def build_embeddings(df: pl.DataFrame) -> dict[str, np.ndarray]:
    return {
        "mean_soap": mean_pool(df, "soap"),
        "mean_mace": mean_pool(df, "mace"),
        "tan_soap": tangent(df, "soap"),
        "tan_mace": tangent(df, "mace"),
    }


# ======================================================================
# 3. QUANTIFICATION  (pure numpy -- no chemistry deps, unit-testable)
# ======================================================================
def cv_knn_r2(X, y, k=15, n_splits=5, n_repeats=5, seed0=0):
    """Out-of-sample kNN-regression R2 (standardised, distance-weighted).
    Repeated K-fold; returns mean, std over repeats."""
    X = np.asarray(X, float)
    y = np.asarray(y, float)
    k = min(k, len(y) - 1)
    scores = []
    for r in range(n_repeats):
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed0 + r)
        pipe = make_pipeline(
            StandardScaler(), KNeighborsRegressor(n_neighbors=k, weights="distance")
        )
        scores.append(cross_val_score(pipe, X, y, cv=kf, scoring="r2").mean())
    return float(np.mean(scores)), float(np.std(scores))


def umap_embed(X, seed, n_neighbors=15, min_dist=0.1):
    Xs = StandardScaler().fit_transform(np.asarray(X, float))
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        metric="euclidean",
        random_state=seed,
    )
    return reducer.fit_transform(Xs)


def umap_knn_r2_over_seeds(X, y, seeds=(0, 1, 2, 3, 4), k=15):
    """UMAP -> 2D -> kNN-R2, once per seed (transductive: UMAP fit on all data,
    kNN cross-validated on the fixed 2D coords). Returns (mean, std, coords_repr).
    coords_repr is the 2D embedding of the seed closest to the mean, for plotting."""
    per_seed, coords = [], []
    for s in seeds:
        emb2d = umap_embed(X, seed=s)
        r2, _ = cv_knn_r2(emb2d, y, k=k, seed0=s)
        per_seed.append(r2)
        coords.append(emb2d)
    per_seed = np.array(per_seed)
    repr_idx = int(np.argmin(np.abs(per_seed - per_seed.mean())))
    return float(per_seed.mean()), float(per_seed.std()), coords[repr_idx]


def pairwise_spearman(X, y):
    """Metric-free ambient baseline: Spearman( pairwise descriptor distance,
    pairwise |d mu| ). Higher => descriptor distance tracks dipole difference."""
    Xs = StandardScaler().fit_transform(np.asarray(X, float))
    dX = pdist(Xs, metric="euclidean")
    dY = pdist(np.asarray(y, float).reshape(-1, 1), metric="cityblock")  # |mu_i - mu_j|
    rho, _ = spearmanr(dX, dY)
    return float(rho)


def morans_i(coords2d, y, k=15):
    """Spatial autocorrelation of |mu| on the UMAP kNN graph (row-standardised
    binary weights). ~0 = random field, ->1 = smooth gradient."""
    coords2d = np.asarray(coords2d, float)
    y = np.asarray(y, float)
    n = len(y)
    k = min(k, n - 1)
    nn = NearestNeighbors(n_neighbors=k + 1).fit(coords2d)
    idx = nn.kneighbors(coords2d, return_distance=False)[:, 1:]
    yc = y - y.mean()
    num = np.sum(yc[:, None] * yc[idx])  # sum_i sum_{j in N(i)} yc_i yc_j
    W = n * k
    den = np.sum(yc ** 2)
    return float((n / W) * (num / den))


def analyze(embeddings: dict[str, np.ndarray], y, seeds=(0, 1, 2, 3, 4), k=15):
    rows, repr_coords = [], {}
    for name, X in embeddings.items():
        amb_mean, amb_std = cv_knn_r2(X, y, k=k)
        um_mean, um_std, coords = umap_knn_r2_over_seeds(X, y, seeds=seeds, k=k)
        rows.append(
            dict(
                representation=name,
                ambient_knn_r2=round(amb_mean, 3),
                ambient_knn_r2_std=round(amb_std, 3),
                umap_knn_r2=round(um_mean, 3),
                umap_knn_r2_std=round(um_std, 3),
                pairwise_spearman=round(pairwise_spearman(X, y), 3),
                morans_i=round(morans_i(coords, y, k=k), 3),
            )
        )
        repr_coords[name] = coords
    return pl.DataFrame(rows), repr_coords


# ======================================================================
# 4. FIGURE
# ======================================================================
def make_figure(repr_coords, y, results, out_png, order=None,
                titles=None):
    order = order or list(repr_coords.keys())
    titles = titles or {
        "mean_soap": "Mean-pooled SOAP",
        "mean_mace": "Mean-pooled MACE",
        "tan_soap": "Tangent (cov) SOAP",
        "tan_mace": "Tangent (cov) MACE",
    }
    rmap = {r["representation"]: r for r in results.to_dicts()}
    fig, axes = plt.subplots(2, 2, figsize=(11, 9.5))
    sc = None
    for ax, name in zip(axes.ravel(), order):
        c = repr_coords[name]
        sc = ax.scatter(c[:, 0], c[:, 1], c=y, cmap="viridis", s=10, alpha=0.85)
        r = rmap[name]
        ax.set_title(
            f"{titles.get(name, name)}\n"
            f"UMAP kNN-$R^2$={r['umap_knn_r2']:.2f}±{r['umap_knn_r2_std']:.2f}  |  "
            f"ambient={r['ambient_knn_r2']:.2f}",
            fontsize=11,
        )
        ax.set_xticks([]); ax.set_yticks([])
    cbar = fig.colorbar(sc, ax=axes.ravel().tolist(), shrink=0.85, pad=0.02)
    cbar.set_label(r"$|\mu|$  (Debye)")
    fig.suptitle("Latent organisation of dipole magnitude on 890 C$_7$H$_{10}$O$_2$ isomers",
                 fontsize=13, y=0.98)
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)


# ======================================================================
# 5. MAIN
# ======================================================================
def run(df: pl.DataFrame, formula="C7H10O2", out_prefix="dipole_latent",
        featurise=False):
    sub = df.filter(pl.col("formula") == formula)
    print(f"{formula}: {sub.height} molecules")

    if featurise:  # only if you don't already have the columns
        sub = add_soap_column(sub)
        sub = add_mace_column(sub)

    y = sub["mu"].to_numpy().astype(float)  # QM9 mu is already |mu| in Debye
    assert np.ndim(y) == 1, "mu must be a scalar magnitude per molecule"

    embeddings = build_embeddings(sub)
    results, repr_coords = analyze(embeddings, y)

    print(results)
    results.write_csv(f"{out_prefix}_metrics.csv")
    make_figure(repr_coords, y, results, f"{out_prefix}_umap.png")
    return results


In [21]:
print(df)

shape: (5_000, 76)
┌───────────┬──────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ mol_id    ┆ formula  ┆ smiles    ┆ canonical ┆ … ┆ soap_embe ┆ soap_matr ┆ mace_embe ┆ mace_matr │
│ ---       ┆ ---      ┆ ---       ┆ _smiles   ┆   ┆ dding     ┆ ix        ┆ dding     ┆ ix        │
│ str       ┆ str      ┆ str       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│           ┆          ┆           ┆ str       ┆   ┆ list[f64] ┆ list[list ┆ list[f64] ┆ list[list │
│           ┆          ┆           ┆           ┆   ┆           ┆ [f64]]    ┆           ┆ [f64]]    │
╞═══════════╪══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ qm9_2     ┆ H2O      ┆ [H]O[H]   ┆ O         ┆ … ┆ [0.014712 ┆ [[0.01136 ┆ [0.151596 ┆ [[0.27217 │
│           ┆          ┆           ┆           ┆   ┆ ,         ┆ 6,        ┆ , -0.1163 ┆ 3, -0.260 │
│           ┆          ┆           ┆           ┆   ┆ 0.043991, ┆ 0.03711

In [22]:
run(df)

2026-06-17 19:57:25.878 | INFO     | src.non_euclidean:_feature_matrices_from_df:48 - Using column: soap_matrix from df
2026-06-17 19:57:25.934 | INFO     | src.non_euclidean:_feature_matrices_from_df:48 - Using column: mace_matrix from df
2026-06-17 19:57:25.981 | INFO     | src.non_euclidean:_feature_matrices_from_df:48 - Using column: soap_matrix from df
2026-06-17 19:57:26.023 | INFO     | src.non_euclidean:matrix_pca:550 - Applying PCA to reduce feature dimension to 17...
2026-06-17 19:57:26.049 | INFO     | src.non_euclidean:matrix_pca:565 - PCA explained variance ratio: 0.9725 (cumulative for 17 components)


C7H10O2: 227 molecules


2026-06-17 19:57:26.100 | INFO     | src.non_euclidean:log_euclidean_vectorize:456 - Smallest eigenvalue across SPD dataset: 6.869264e-04
2026-06-17 19:57:26.100 | INFO     | src.non_euclidean:log_euclidean_vectorize:464 - Minimum eigenvalue looks structurally stable.
2026-06-17 19:57:26.100 | INFO     | src.non_euclidean:_feature_matrices_from_df:48 - Using column: mace_matrix from df
2026-06-17 19:57:26.142 | INFO     | src.non_euclidean:matrix_pca:550 - Applying PCA to reduce feature dimension to 17...
2026-06-17 19:57:26.154 | INFO     | src.non_euclidean:matrix_pca:565 - PCA explained variance ratio: 0.9867 (cumulative for 17 components)
2026-06-17 19:57:26.198 | INFO     | src.non_euclidean:log_euclidean_vectorize:456 - Smallest eigenvalue across SPD dataset: 8.725806e-02
2026-06-17 19:57:26.198 | INFO     | src.non_euclidean:log_euclidean_vectorize:464 - Minimum eigenvalue looks structurally stable.
/Users/karlfindhansen/thesis/Anomaly-Detection-in-Molecular-and-Materials-Datase

shape: (4, 7)
┌──────────────┬──────────────┬──────────────┬─────────────┬──────────────┬─────────────┬──────────┐
│ representati ┆ ambient_knn_ ┆ ambient_knn_ ┆ umap_knn_r2 ┆ umap_knn_r2_ ┆ pairwise_sp ┆ morans_i │
│ on           ┆ r2           ┆ r2_std       ┆ ---         ┆ std          ┆ earman      ┆ ---      │
│ ---          ┆ ---          ┆ ---          ┆ f64         ┆ ---          ┆ ---         ┆ f64      │
│ str          ┆ f64          ┆ f64          ┆             ┆ f64          ┆ f64         ┆          │
╞══════════════╪══════════════╪══════════════╪═════════════╪══════════════╪═════════════╪══════════╡
│ mean_soap    ┆ -0.042       ┆ 0.044        ┆ -0.125      ┆ 0.028        ┆ -0.029      ┆ 0.016    │
│ mean_mace    ┆ 0.264        ┆ 0.02         ┆ 0.229       ┆ 0.016        ┆ 0.061       ┆ 0.277    │
│ tan_soap     ┆ -0.012       ┆ 0.027        ┆ -0.077      ┆ 0.017        ┆ -0.006      ┆ 0.033    │
│ tan_mace     ┆ 0.331        ┆ 0.013        ┆ 0.287       ┆ 0.019        ┆ 0

representation,ambient_knn_r2,ambient_knn_r2_std,umap_knn_r2,umap_knn_r2_std,pairwise_spearman,morans_i
str,f64,f64,f64,f64,f64,f64
"""mean_soap""",-0.042,0.044,-0.125,0.028,-0.029,0.016
"""mean_mace""",0.264,0.02,0.229,0.016,0.061,0.277
"""tan_soap""",-0.012,0.027,-0.077,0.017,-0.006,0.033
"""tan_mace""",0.331,0.013,0.287,0.019,0.086,0.349
